In [ ]:
!pip install kss
!apt-get -qq install mecab libmecab-dev mecab-ko mecab-ko-dic
!pip install mecab-python3
!pip install imbalanced-learn
!pip install -U sentence-transformers
!pip install imblearn
!pip install pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 55.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 12.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.3/131.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.9/757.9 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.4/939.4 k

^C
^C


In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import random

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/DataSet/네이버포털_전처리완료.csv", encoding = "utf-8-sig")

In [ ]:
df.head()

,review,data
0,꽃가루 알러지 없는 부모님은 모르는 아이의 꽃가루 알러지 시작되었습니다 이 글을 봄...,2022-04-10
1,꽃가루 알러지 있는 자녀 두신 부모님 보세요 네이버 꽃가루 지수 검색하면 꽃가루 소...,2021-05-09
2,부산 유아 아토피 한의학에서 바라보는 원인과 증상에 대하여 가려움에 밤잠 못 이루는...,2025-04-14
3,꽃가루 알러지 있으신 자녀 키우시는 분들 보세요 학기초만 되면 보통 중간고사 기간부...,2021-04-18
4,어린이 운동회에 참여하다 이제 겨울이 가고 봄이 찾아왔다 봄이란 무엇인가 봄은 본디...,2024-04-19


**섞어서 임의로 5만건 추출**

In [ ]:
shuffled_df = df.sample(frac=1).reset_index(drop=True)

## LLM - 5만건 테스트/훈련셋 분리해서 만들기
각단계 참조

##공통수행 전처리 1단계


### kss 문장 **분리**

In [ ]:
import pandas as pd
from kss import split_sentences
from tqdm import tqdm

tqdm.pandas()
# 텍스트가 담긴 컬럼명 (실제 이름에 맞게 수정)
text_column = "review"
# 문장 분리
shuffled_df["sentences"] = shuffled_df[text_column].progress_apply(lambda x: split_sentences(x, backend='mecab'))


ModuleNotFoundError: No module named 'kss'

In [ ]:
shuffled_df.to_csv("/content/drive/MyDrive/DataSet/shuffled_df_sentences.csv", encoding = "utf-8-sig")

### 1차 Rule 기반 광고, 소설, 홍보글 제거
: 라벨링 단계

In [ ]:
import pandas as pd

# 데이터 불러오기
df = pd.read_csv("/content/drive/MyDrive/DataSet/shuffled_df_sentences.csv", encoding="cp949")

# --- 키워드 사전 정의 ---
person_keywords = [
    "나는", "제가", "저는", "우리", "남편", "아내", "아이", "딸", "아들",
    "힘들", "고민", "좋아하", "싫어하", "사랑", "짜증", "속상", "행복", "걱정", "눈물"
]

fiction_keywords = [
    "그는", "그녀는", "되뇌었다", "속삭였다", "눈물이", "바라보며", "기억했다",
    "미소를 지었다", "한참을", "창밖을", "어둠", "사라졌다", "느껴졌다"
]

religious_keywords = [
    "하나님", "예수", "성령", "회개", "지옥", "천국", "복음", "기도", "전도", "영혼", "창조",
    "죄", "구원", "용서", "믿음", "십자가", "사탄", "말씀", "믿으십시오", "사람은 죄인입니다",
    "말세", "말세지말", "마귀", "다시 태어나", "지음받아야", "영적", "부활", "기독교"
]

ad_keywords = [
    "상품", "옵션", "ml", "세트", "강화유리", "액정보호필름", "보호필름", "갤럭시탭", "최저가격", "적립", "배송"
]

system_messages = [
    "문제가 발생했습니다", "다시 시도해주세요", "페이지를 찾을 수 없습니다", "시스템 오류", "불편을 드려 죄송합니다"
]

url_keywords = [
    "http", "https", "naver", "com", "smartstore", "storefarm", "kindlycare", "blog", "카인들리케어",
    "네이버쇼핑", "스마트스토어", "브랜드", "공식몰", "kr", "소소한 일상에 행복을", "인스타그램"
]

ad_descriptive_keywords = [
    "장점", "단점", "기능", "특징", "효과적", "효율적", "성능", "방수", "방풍", "편안함",
    "내구성", "디자인", "착용감", "활동성", "추천", "최적", "우수", "차단", "제공",
    "즐길 수 있습니다", "완벽한", "보장", "우수한", "탁월한"
]

msg_request_keywords = [
    "쪽지", "쪽찌", "보냈", "보낼께", "부탁드려", "성함", "업체", "정보", "알려주", "알수있"
]

encyclopedic_keywords = [
    "설명", "의미", "정의", "특징", "성분", "화학식", "연구", "사용", "효능", "발견", "구성",
    "기원", "기술", "재료", "원리", "효과", "원소", "명칭", "개념", "유래", "발생"
]

# --- 1차 라벨링 ---
def is_ad_sentence(sentence: str) -> bool:
    return any(kw in sentence for kw in ad_keywords)

def is_person_sentence(sentence: str) -> bool:
    return any(kw in sentence for kw in person_keywords)

def initial_label(sentence: str) -> int:
    if is_ad_sentence(sentence):
        return 0
    elif is_person_sentence(sentence):
        return 1
    else:
        return 1  # 기본값

df['label'] = df['sentence'].astype(str).apply(initial_label)

# --- 문서 단위 재분류 ---
grouped = df[df['label'] == 1].groupby("doc_id")

fiction_doc_ids = {
    doc_id for doc_id, group in grouped
    if sum(any(kw in str(s) for kw in fiction_keywords) for s in group['sentence']) >= 4
}

religious_doc_ids = {
    doc_id for doc_id, group in grouped
    if sum(any(kw in str(s) for kw in religious_keywords) for s in group['sentence']) >= 5
}

# --- 문장 단위 세부 재분류 ---
def is_encyclopedia(text):
    text = str(text)
    return sum(kw in text for kw in encyclopedic_keywords) >= 10

def relabel(row):
    label = row['label']
    text = str(row['sentence']).lower()
    doc_id = row['doc_id']

    if label != 1:
        return label
    if doc_id in fiction_doc_ids:
        return 2
    if doc_id in religious_doc_ids:
        return 4
    if any(msg in text for msg in system_messages):
        return 5
    if any(kw in text for kw in url_keywords):
        return 6
    if sum(kw in text for kw in ad_descriptive_keywords) >= 2:
        return 7
    if is_encyclopedia(text):
        return 9
    return 1

df['label'] = df.apply(relabel, axis=1)

# --- 쪽지/연락 유도 문장 → 8 ---
def is_label_8_candidate(text):
    text = str(text)
    if "쪽" in text or "성함" in text or "업체" in text or "정보" in text:
        count = sum(kw in text for kw in msg_request_keywords)
        return count >= 2
    return False

df['label'] = df.apply(
    lambda row: 8 if row['label'] == 1 and is_label_8_candidate(row['sentence']) else row['label'],
    axis=1
)

# 저장
df.to_csv("labeled_final_output.csv", index=False, encoding="cp949")


2차적 정제 rule 기반 데이터 정제

In [ ]:
import pandas as pd
import re

# --- 데이터 불러오기 ---
df = pd.read_csv("/content/drive/MyDrive/DataSet/shuffled_df_flat.csv")

# --- 키워드 사전 정의 ---
person_keywords = [
    "나는", "제가", "저는", "우리", "남편", "아내", "아이", "딸", "아들",
    "힘들", "고민", "좋아하", "싫어하", "사랑", "짜증", "속상", "행복", "걱정", "눈물"
]

fiction_keywords = [
    "그는", "그녀는", "되뇌었다", "속삭였다", "눈물이", "바라보며", "기억했다",
    "미소를 지었다", "한참을", "창밖을", "어둠", "사라졌다", "느껴졌다"
]

religious_keywords = [
    "하나님", "예수", "성령", "회개", "지옥", "천국", "복음", "기도", "전도", "영혼", "창조",
    "죄", "구원", "용서", "믿음", "십자가", "사탄", "말씀", "믿으십시오", "사람은 죄인입니다",
    "말세", "말세지말", "마귀", "다시 태어나", "지음받아야", "영적", "부활", "기독교"
]

ad_keywords = [
    "상품", "옵션", "ml", "세트", "강화유리", "액정보호필름", "보호필름", "갤럭시탭", "최저가격", "적립", "배송"
]

system_messages = [
    "문제가 발생했습니다", "다시 시도해주세요", "페이지를 찾을 수 없습니다", "시스템 오류", "불편을 드려 죄송합니다"
]

url_keywords = [
    "http", "https", "naver", "com", "smartstore", "storefarm", "kindlycare", "blog", "카인들리케어",
    "네이버쇼핑", "스마트스토어", "브랜드", "공식몰", "kr", "소소한 일상에 행복을", "인스타그램"
]

ad_descriptive_keywords = [
    "장점", "단점", "기능", "특징", "효과적", "효율적", "성능", "방수", "방풍", "편안함",
    "내구성", "디자인", "착용감", "활동성", "추천", "최적", "우수", "차단", "제공",
    "즐길 수 있습니다", "완벽한", "보장", "우수한", "탁월한"
]

msg_request_keywords = [
    "쪽지", "쪽찌", "보냈", "보낼께", "부탁드려", "성함", "업체", "정보", "알려주", "알수있"
]

encyclopedic_keywords = [
    "설명", "의미", "정의", "특징", "성분", "화학식", "연구", "사용", "효능", "발견", "구성",
    "기원", "기술", "재료", "원리", "효과", "원소", "명칭", "개념", "유래", "발생"
]

# --- 의미 없음 판단 ---
def is_meaningless(text):
    if pd.isna(text):
        return True
    text = str(text).strip().lower()
    meaningless_phrases = [
        "네", "예", "응", "음", "오", "헐", "헉", "ㅋㅋ", "ㅠㅠ", "ㅎㅎ", "굿", "넵",
        "감사", "좋아요", "좋아", "감사합니다", "고마워요", "맞아요", "마자요", "저도요",
        "공감", "수산뇨", "추천해요", "작아요", "무서워요", "힘내세요", "어찌까요", "미투요",
        "왜 안돼", "그러게요", "밧이요", "보내드라", "수이매냐", "매냐", "그립다", "했죠", "안그래요",
        "내용없음", "없음", "내용 없습니다", "해당 없음", "n/a", "none", "-", " "
    ]
    if len(text) <= 5:
        return True
    if any(text == phrase for phrase in meaningless_phrases):
        return True
    if re.fullmatch(r"[.!\?~ㄷㅎㅋㅜㅠ\s]+", text):
        return True
    return False

# --- 1차 라벨링 ---
def is_ad_sentence(sentence: str) -> bool:
    return any(kw in sentence for kw in ad_keywords)

def is_person_sentence(sentence: str) -> bool:
    return any(kw in sentence for kw in person_keywords)

def initial_label(sentence: str) -> int:
    if is_meaningless(sentence):
        return -1
    if is_ad_sentence(sentence):
        return 0
    elif is_person_sentence(sentence):
        return 1
    else:
        return 1

df['label'] = df['sentence'].astype(str).apply(initial_label)

# --- 문서 단위 재분류 (개선된 소설 및 종교 판단 포함) ---
grouped = df[df['label'] == 1].groupby("doc_id")

# ✅ 개선된 소설 판별 로직
def is_fiction_doc(group):
    count = sum(
        sum(kw in str(s) for kw in fiction_keywords) >= 2
        for s in group['sentence']
    )
    return count >= 5


fiction_doc_ids = {
    doc_id for doc_id, group in grouped
    if is_fiction_doc(group)
}

# ✅ 종교 문서 판단 로직
def is_religious_doc(group):
    count = sum(
        1 for s in group['sentence']
        if sum(kw in str(s) for kw in religious_keywords) >= 2
    )
    return count >= 5

religious_doc_ids = {
    doc_id for doc_id, group in grouped
    if is_religious_doc(group)
}

# --- 문장 단위 재분류 ---
def is_encyclopedia(text):
    text = str(text)
    return sum(kw in text for kw in encyclopedic_keywords) >= 10

def relabel(row):
    label = row['label']
    text = str(row['sentence']).lower()
    doc_id = row['doc_id']

    if label != 1:
        return label
    if doc_id in fiction_doc_ids:
        return 2
    if doc_id in religious_doc_ids:
        return 4
    if any(msg in text for msg in system_messages):
        return 5
    if any(kw in text for kw in url_keywords):
        return 6
    if sum(kw in text for kw in ad_descriptive_keywords) >= 5:
        return 7
    if is_encyclopedia(text):
        return 9
    return 1

df['label'] = df.apply(relabel, axis=1)

# --- 쪽지 유도 → 8 ---
def is_label_8_candidate(text):
    text = str(text)
    if "쪽" in text or "성함" in text or "업체" in text or "정보" in text:
        count = sum(kw in text for kw in msg_request_keywords)
        return count >= 2
    return False

df['label'] = df.apply(
    lambda row: 8 if row['label'] == 1 and is_label_8_candidate(row['sentence']) else row['label'],
    axis=1
)

# --- 결과 저장 ---
df.to_csv("labeled_final_output.csv", index=False, encoding="cp949")


In [ ]:
import pandas as pd
import re

def apply_custom_labeling(df: pd.DataFrame) -> pd.DataFrame:
    # --- 키워드 사전 정의 ---
    person_keywords = [
        "나는", "제가", "저는", "우리", "남편", "아내", "아이", "딸", "아들",
        "힘들", "고민", "좋아하", "싫어하", "사랑", "짜증", "속상", "행복", "걱정", "눈물"
    ]
    fiction_keywords = [
        "그는", "그녀는", "되뇌었다", "속삭였다", "눈물이", "바라보며", "기억했다",
        "미소를 지었다", "한참을", "창밖을", "어둠", "사라졌다", "느껴졌다"
    ]
    religious_keywords = [
        "하나님", "예수", "성령", "회개", "지옥", "천국", "복음", "기도", "전도", "영혼", "창조",
        "죄", "구원", "용서", "믿음", "십자가", "사탄", "말씀", "믿으십시오", "사람은 죄인입니다",
        "말세", "말세지말", "마귀", "다시 태어나", "지음받아야", "영적", "부활", "기독교"
    ]
    ad_keywords = ["상품", "옵션", "ml", "세트", "강화유리", "액정보호필름", "보호필름", "갤럭시탭", "최저가격", "적립", "배송"]
    system_messages = ["문제가 발생했습니다", "다시 시도해주세요", "페이지를 찾을 수 없습니다", "시스템 오류", "불편을 드려 죄송합니다"]
    url_keywords = ["http", "https", "naver", "com", "smartstore", "storefarm", "kindlycare", "blog", "카인들리케어",
                    "네이버쇼핑", "스마트스토어", "브랜드", "공식몰", "kr", "소소한 일상에 행복을", "인스타그램"]
    ad_descriptive_keywords = [
        "장점", "단점", "기능", "특징", "효과적", "효율적", "성능", "방수", "방풍", "편안함",
        "내구성", "디자인", "착용감", "활동성", "추천", "최적", "우수", "차단", "제공",
        "즐길 수 있습니다", "완벽한", "보장", "우수한", "탁월한"
    ]
    msg_request_keywords = ["쪽지", "쪽찌", "보냈", "보낼께", "부탁드려", "성함", "업체", "정보", "알려주", "알수있"]
    encyclopedic_keywords = [
        "설명", "의미", "정의", "특징", "성분", "화학식", "연구", "사용", "효능", "발견", "구성",
        "기원", "기술", "재료", "원리", "효과", "원소", "명칭", "개념", "유래", "발생"
    ]
    meaningless_phrases = [
        "네", "예", "응", "음", "오", "헐", "헉", "ㅋㅋ", "ㅠㅠ", "ㅎㅎ", "굿", "넵",
        "감사", "좋아요", "좋아", "감사합니다", "고마워요", "맞아요", "마자요", "저도요",
        "공감", "수산뇨", "추천해요", "작아요", "무서워요", "힘내세요", "어찌까요", "미투요",
        "왜 안돼", "그러게요", "밧이요", "보내드라", "수이매냐", "매냐", "그립다", "했죠", "안그래요",
        "내용없음", "없음", "내용 없습니다", "해당 없음", "n/a", "none", "-", " "
    ]

    # --- 라벨링 보조 함수 ---
    def is_meaningless(text):
        if pd.isna(text):
            return True
        text = str(text).strip().lower()
        if len(text) <= 5:
            return True
        if any(text == phrase for phrase in meaningless_phrases):
            return True
        if re.fullmatch(r"[.!\?~ㄷㅎㅋㅜㅠ\s]+", text):
            return True
        return False

    def initial_label(sentence: str) -> int:
        if is_meaningless(sentence):
            return -1
        if any(kw in sentence for kw in ad_keywords):
            return 0
        elif any(kw in sentence for kw in person_keywords):
            return 1
        else:
            return 1

    df['label'] = df['sentence'].astype(str).apply(initial_label)

    # --- 문서 단위 그룹 재분류 ---
    grouped = df[df['label'] == 1].groupby("doc_id")

    def is_fiction_doc(group):
        count = sum(
            sum(kw in str(s) for kw in fiction_keywords) >= 2
            for s in group['sentence']
        )
        return count >= 5

    def is_religious_doc(group):
        count = sum(
            1 for s in group['sentence']
            if sum(kw in str(s) for kw in religious_keywords) >= 2
        )
        return count >= 5

    fiction_doc_ids = {doc_id for doc_id, group in grouped if is_fiction_doc(group)}
    religious_doc_ids = {doc_id for doc_id, group in grouped if is_religious_doc(group)}

    def is_encyclopedia(text):
        return sum(kw in str(text) for kw in encyclopedic_keywords) >= 6

    def relabel(row):
        label, text, doc_id = row['label'], str(row['sentence']).lower(), row['doc_id']
        if label != 1:
            return label
        if doc_id in fiction_doc_ids:
            return 2
        if doc_id in religious_doc_ids:
            return 4
        if any(msg in text for msg in system_messages):
            return 5
        if any(kw in text for kw in url_keywords):
            return 6
        if sum(kw in text for kw in ad_descriptive_keywords) >= 5:
            return 7
        if is_encyclopedia(text):
            return 9
        return 1

    df['label'] = df.apply(relabel, axis=1)

    def is_label_8_candidate(text):
        text = str(text)
        if "쪽" in text or "성함" in text or "업체" in text or "정보" in text:
            return sum(kw in text for kw in msg_request_keywords) >= 2
        return False

    df['label'] = df.apply(
        lambda row: 8 if row['label'] == 1 and is_label_8_candidate(row['sentence']) else row['label'],
        axis=1
    )
    mask = (
    df["sentence"].str.fullmatch(r'[A-Za-z ]+', na=False) &  # 영어 문장만
    (df["label"] == 1)                                       # label이 1인 경우만
    )

    df.loc[mask, "label"] = 10

    return df


## 문장 전처리 구성
1. shuffled_df_sentences : 전체 데이터 중에서 5만건 랜덤 선정
2. shuffled_df_sentences : 데이터를 훈련셋 fit에 맞춰서 분리

In [ ]:
shuffled_df = pd.read_csv("/content/drive/MyDrive/DataSet/shuffled_df_sentences.csv", encoding = "utf-8-sig")

In [ ]:
import re

def fast_token_parser(s):
    return re.findall(r"'(.*?)'", s)

In [ ]:
import pandas as pd
import numpy as np

문장 리스트화 하고 문장단위로 분리

In [ ]:
rows = []

for doc_idx, row in shuffled_df.iterrows():
    sentences = row["sentences_clean"]
    for idx, sentence in enumerate(sentences):
        rows.append({
            "sentence": sentence,
            "doc_id": doc_idx,
            "sentence_idx": idx
        })

flat_df = pd.DataFrame(rows)


In [ ]:
print(flat_df.head())
print(len(flat_df))

                                            sentence  doc_id  sentence_idx
0  The Summer Days 단편 소개팅해서 개월 좀 넘게 만났고 헤어진지는 주 정...       0             0
1  둘다 살 동갑이고 제가 좋아하는것보다 본인이 절 덜 좋아하는거 같다길래 의지가 있는...       0             1
2        그리고 더는 자신이 없다는 얘기를 듣고 결국 잘지내자는 말을 끝으로 끝났습니다       0             2
3  우선 전남친이 불만이었고 화냈던 포인트를 말하자면 뭔가 본인이 원하는 이상형이 뚜렷...       0             3
4  머리 스타일이라던지 옷입는 취향이라던지 제가 추위를 좀 많이 타는 편이라 겨울에 코...       0             4
3799808


In [ ]:
#flat_df 저장하기
flat_df.to_csv("/content/drive/MyDrive/DataSet/shuffled_df_flat.csv")

kss 문장 분리 실행기가 너무 느려서 실행한것

In [ ]:
#라벨링을 위해서 분할해서 저장하기

# 1. 원본 CSV 파일 경로
file_path = '/content/drive/MyDrive/DataSet/shuffled_df_flat.csv'

# 파일 이름을 필요에 따라 수정하세요

# 2. CSV 파일 불러오기 (인코딩 자동 선택)
encodings = ['utf-8', 'cp949', 'ISO-8859-1']
for enc in encodings:
    try:
        df = pd.read_csv(file_path, encoding=enc)
        break
    except Exception as e:
        continue

# 3. 총 행 수 기준으로 5개로 균등 분할
chunk_size = len(df) // 5
chunks = [df.iloc[i * chunk_size: (i + 1) * chunk_size] for i in range(4)]
chunks.append(df.iloc[4 * chunk_size:])  # 마지막 chunk는 남은 모든 행 포함

# 4. 각각 저장
for i, chunk in enumerate(chunks):
    output_path = f'/content/drive/MyDrive/DataSet/shuffled_part_{i+1}.csv'
    chunk.to_csv(output_path, index=False, encoding='utf-8')
    print(f'Saved: {output_path}')


Saved: /content/drive/MyDrive/DataSet/shuffled_part_1.csv
Saved: /content/drive/MyDrive/DataSet/shuffled_part_2.csv
Saved: /content/drive/MyDrive/DataSet/shuffled_part_3.csv
Saved: /content/drive/MyDrive/DataSet/shuffled_part_4.csv
Saved: /content/drive/MyDrive/DataSet/shuffled_part_5.csv


위에 라벨링을 위한  rule 기반 파싱 함수 적용

In [ ]:
labeled_df = apply_custom_labeling(flat_df)

In [ ]:
labeled_df.to_csv("/content/drive/MyDrive/DataSet/suffled_sample/labeled_final_output.csv", index=False, encoding="cp949")

### 무의미한 rule 기반 제거가능한 문자 제거

In [ ]:
labels_to_remove = [-1, 10, 6, 8, 5]

# 해당 라벨이 아닌 행만 남기기
filtered_df = labeled_df[~labeled_df['label'].isin(labels_to_remove)]

# 결과 확인 (선택)
print(filtered_df['label'].value_counts())

# 필요 시 저장
filtered_df.to_csv("/content/drive/MyDrive/DataSet/suffled_sample/filtered_output.csv", index=False, encoding="utf-8-sig")

label
1    2838724
0     421352
4     317385
2      79313
7        174
9         63
Name: count, dtype: int64


In [ ]:
def map_to_merged_label(label):
    if label in [0, 7]:
        return 0  # 광고성
    elif label == 1:
        return 1  # 사용자 감성 표현
    elif label == 2:
        return 2  # 픽션(문학적 문체)
    elif label == 4:
        return 3  # 종교적 정보
    elif label == 9:
        return 4  # 백과사전 식 정보글
    else:
        return -1  # 예외 처리용 (필요 시 필터링 가능)

# 새로운 컬럼으로 저장
train_df['label'] = train_df['label'].apply(map_to_merged_label)

cross vaildation 검증시 안해도 됨

In [ ]:
from sklearn.model_selection import train_test_split
# 문서 단위로 나누기
doc_ids = filtered_df["doc_id"].unique()
train_ids, test_ids = train_test_split(doc_ids, test_size=0.2, random_state=42)

train_df = filtered_df[filtered_df["doc_id"].isin(train_ids)]
test_df = filtered_df[filtered_df["doc_id"].isin(test_ids)]

In [ ]:
train_df.to_csv("/content/drive/MyDrive/DataSet/suffled_sample/train.csv", index=False)
test_df.to_csv("/content/drive/MyDrive/DataSet/suffled_sample/test.csv", index=False)

NameError: name 'train_df' is not defined

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
train_df= pd.read_csv("/content/drive/MyDrive/DataSet/suffled_sample/train.csv")
test_df = pd.read_csv("/content/drive/MyDrive/DataSet/suffled_sample/test.csv")

In [ ]:
test_df["merged_label"].value_counts()

,count
merged_label,
1,580718
0,84223
3,65241
2,17614
4,15


##임베딩 - kobert
소주점 tolist 변환과정에서 미세한 소수점 차이 존재

train/test

In [ ]:
train_df= pd.read_csv("/content/drive/MyDrive/DataSet/suffled_sample/train.csv")
test_df = pd.read_csv("/content/drive/MyDrive/DataSet/suffled_sample/test.csv")

In [ ]:
from imblearn.over_sampling import SMOTE, RandomOverSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import random
import os

# 디바이스 설정
import torch

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"


model = SentenceTransformer('jhgan/ko-sroberta-multitask', device=device)

# 임베딩 대상 문장 리스트
embedding = train_df["sentence"].tolist()
y = train_df['merged_label'].values

# 중간 저장 폴더
save_dir = "/content/drive/MyDrive/DataSet/suffled_sample"
os.makedirs(save_dir, exist_ok=True)

# 임베딩 및 중간 저장
batch_size=5000
X_all = []
model.eval()
for i in range(0, len(embedding), batch_size):
    batch_sentences = embedding[i:i+batch_size]
    X_batch = model.encode(batch_sentences, batch_size=5000, convert_to_numpy=True, normalize_embeddings=False, show_progress_bar=True)
    X_all.extend(X_batch)

    # 중간 저장: numpy 형식으로 저장
    np.save(os.path.join(save_dir, f"train_batch1000_{i//batch_size:04d}.npy"), X_batch)
    print(f"[{i}/{len(embedding)}] 저장됨")

# 모든 임베딩 합치기
X = np.vstack(X_all)

# 필요하다면 같이 저장
np.save(os.path.join(save_dir, "train_X_final.npy"), X)
pd.Series(y).to_csv(os.path.join(save_dir, "train_y_labels.csv"), index=False)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[0/2909200] 저장됨


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[5000/2909200] 저장됨


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

[2895000/2909200] 저장됨


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[2900000/2909200] 저장됨


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[2905000/2909200] 저장됨


In [ ]:
train_df["embedding"] = list(X)

In [ ]:
test_df["merged_label"].value_counts()

,count
merged_label,
1,580718
0,84223
3,65241
2,17614
4,15


In [ ]:
Totall_Data = pd.concat([train_df, test_df])

In [ ]:
Totall_Data["merged_label"].value_counts()

,count
merged_label,
1,2838724
0,421526
3,317385
2,79313
4,63


In [ ]:
import torch, gc
gc.collect()  # Python 객체 정리
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

###중간에 멈추는 경우

In [ ]:
from imblearn.over_sampling import SMOTE, RandomOverSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import os
# 디바이스 설정
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

# 중간 저장 폴더
save_dir = "/content/drive/MyDrive/DataSet/suffled_sample"
os.makedirs(save_dir, exist_ok=True)

# 모델 로드
model = SentenceTransformer('jhgan/ko-sroberta-multitask', device=device)

start_index = 284963  # 시작 인덱스
batch_size = 32

# 부분 리스트 및 레이블 슬라이싱
embedding = train_df["sentence"].tolist()[start_index:]
y = train_df['merged_label'].values[start_index:]

X_all = []
for i in range(0, len(embedding), batch_size):
    batch_sentences = embedding[i:i+batch_size]
    X_batch = model.encode(batch_sentences, show_progress_bar=False)
    X_all.extend(X_batch)

    # 저장: 전체 인덱스를 기준으로 파일명 생성
    global_index = start_index + i
    np.save(os.path.join(save_dir, f"batch_{global_index//batch_size:04d}.npy"), X_batch)
    print(f"[{global_index}/{start_index + len(embedding)}] 저장됨")
    print((global_index//batch_size))
    if (global_index//batch_size) == 8978:
        break

# 임베딩 병합 및 저장
X = np.vstack(X_all)
np.save(os.path.join(save_dir, f"X_final_from_{start_index}.npy"), X)
pd.Series(y).to_csv(os.path.join(save_dir, f"y_labels_from_{start_index}.csv"), index=False)


[284963/2909200] 저장됨
8905
[284995/2909200] 저장됨
8906
[285027/2909200] 저장됨
8907
[285059/2909200] 저장됨
8908
[285091/2909200] 저장됨
8909
[285123/2909200] 저장됨
8910
[285155/2909200] 저장됨
8911
[285187/2909200] 저장됨
8912
[285219/2909200] 저장됨
8913
[285251/2909200] 저장됨
8914
[285283/2909200] 저장됨
8915
[285315/2909200] 저장됨
8916
[285347/2909200] 저장됨
8917
[285379/2909200] 저장됨
8918
[285411/2909200] 저장됨
8919
[285443/2909200] 저장됨
8920
[285475/2909200] 저장됨
8921
[285507/2909200] 저장됨
8922
[285539/2909200] 저장됨
8923
[285571/2909200] 저장됨
8924
[285603/2909200] 저장됨
8925
[285635/2909200] 저장됨
8926
[285667/2909200] 저장됨
8927
[285699/2909200] 저장됨
8928
[285731/2909200] 저장됨
8929
[285763/2909200] 저장됨
8930
[285795/2909200] 저장됨
8931
[285827/2909200] 저장됨
8932
[285859/2909200] 저장됨
8933
[285891/2909200] 저장됨
8934
[285923/2909200] 저장됨
8935
[285955/2909200] 저장됨
8936
[285987/2909200] 저장됨
8937
[286019/2909200] 저장됨
8938
[286051/2909200] 저장됨
8939
[286083/2909200] 저장됨
8940
[286115/2909200] 저장됨
8941
[286147/2909200] 저장됨
8942
[286179/2909

병합하기

In [ ]:
import os
import numpy as np

# 저장된 배치 파일 경로
save_dir = "/content/drive/MyDrive/DataSet/suffled_sample"

# 모든 batch 파일 정렬
npy_files = sorted(
    [f for f in os.listdir(save_dir) if f.startswith("batch_") and f.endswith(".npy")],
    key=lambda x: int(x.split("_")[1].split(".")[0])
)

# ✅ 현재 파트 설정 (세션마다 다르게 바꿔야 함!)
part_idx = 5
files_per_part = 10000

# 파트 범위 설정
start = part_idx * files_per_part
end = start + files_per_part
npy_files_part = npy_files[start:end]

# 병합
X_all = []
for fname in npy_files_part:
    batch = np.load(os.path.join(save_dir, fname))
    X_all.append(batch)
    print(f"{fname} 로드됨")

X_merged = np.vstack(X_all)
print(f"[파트 {part_idx}] 병합 shape: {X_merged.shape}")

# 저장
np.save(os.path.join(save_dir, f"X_final_part_{part_idx}.npy"), X_merged)


KeyboardInterrupt: 

다시 한번 한개로 만들기

In [ ]:
import os
import numpy as np

# 저장 경로
save_dir = "/content/drive/MyDrive/DataSet/suffled_sample"

# part 파일 이름 생성 (0~8)
part_files = [f"X_final_part_{i}.npy" for i in range(10)]

# 병합
X_parts = []
for fname in part_files:
    path = os.path.join(save_dir, fname)
    print(f"🔄 {fname} 로딩 중...")
    X = np.load(path)
    X_parts.append(X)
    print(f"✅ shape: {X.shape}")

# 전체 병합
X_all = np.vstack(X_parts)
print(f"🎉 전체 병합 shape: {X_all.shape}")

# 최종 저장
np.save(os.path.join(save_dir, "X_final_all.npy"), X_all)
print("💾 X_final_all.npy 저장 완료")


🔄 X_final_part_0.npy 로딩 중...
✅ shape: (320000, 768)
🔄 X_final_part_1.npy 로딩 중...
✅ shape: (320000, 768)
🔄 X_final_part_2.npy 로딩 중...
✅ shape: (320000, 768)
🔄 X_final_part_3.npy 로딩 중...
✅ shape: (320000, 768)
🔄 X_final_part_4.npy 로딩 중...
✅ shape: (320000, 768)
🔄 X_final_part_5.npy 로딩 중...
✅ shape: (320000, 768)
🔄 X_final_part_6.npy 로딩 중...
✅ shape: (320000, 768)
🔄 X_final_part_7.npy 로딩 중...
✅ shape: (320000, 768)
🔄 X_final_part_8.npy 로딩 중...
✅ shape: (320000, 768)
🔄 X_final_part_9.npy 로딩 중...
✅ shape: (29200, 768)
🎉 전체 병합 shape: (2909200, 768)
💾 X_final_all.npy 저장 완료


In [ ]:
train = np.load("/content/drive/MyDrive/DataSet/suffled_sample/X_final_all.npy")
test = np.load("/content/drive/MyDrive/DataSet/suffled_sample/test_X_final.npy")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/DataSet/suffled_sample/test_X_final_all.npy'

In [ ]:
train_df["embedding"] = list(train)
test_df["embedding"] = list(test)

In [ ]:
train_df["merged_label"].value_counts()

,count
merged_label,
1,2258006
0,337303
3,252144
2,61699
4,48


In [ ]:
train_totall = pd.concat([train_df, test_df])

In [ ]:
len(train_totall)

3657011

In [ ]:
emb1 = model.encode(test_df["sentence"][100], convert_to_numpy=True)
emb2 = model.encode(test_df["sentence"][100], convert_to_numpy=True)

np.allclose(emb1, emb2, atol=1e-6)  # True여야 정상

True

In [ ]:
matching_idx = test_df.index[
   test_df["embedding"].apply(lambda x: np.array_equal(x, X_batch))
].tolist()

print(matching_idx)  # 예: [1]

[]


In [ ]:
model = SentenceTransformer('jhgan/ko-sroberta-multitask', device=device)

In [ ]:
test_df["sentence"][100]

'상품 후기보기유통기한 사용기한 표시의무 대상 상품은 사용기한 이후인 상품 제조국 한국옵션 아토팜 수딩 젤 로션 ml 개 PSP 모두의 골프 포터블 최저가격 원최대 원 적립총 분이상이 개를 주셨습니다'

In [ ]:
X_batch = model.encode(test_df["sentence"][100], show_progress_bar=False)

In [ ]:
X_batch

array([ 1.41409725e-01,  2.31301904e-01,  2.12527230e-01,  8.63969997e-02,
        3.74197006e-01, -5.60273111e-01,  9.50763971e-02, -3.84796737e-03,
        8.66491914e-01, -2.20726252e-01,  4.14837182e-01, -2.25773573e-01,
       -3.97686690e-01,  1.77608296e-01,  2.69867450e-01,  2.46545877e-02,
        3.66385132e-02,  1.67650685e-01, -3.25799286e-01, -3.78762692e-01,
       -8.07964385e-01, -5.35096586e-01,  8.41159374e-02, -2.33489215e-01,
       -3.28113139e-01, -1.26334190e-01, -2.95861334e-01,  4.66363765e-02,
        1.06387055e+00,  5.08943051e-02,  4.03666645e-01,  4.86346073e-02,
       -1.07323825e+00, -2.88435578e-01,  3.15351009e-01, -2.64225513e-01,
       -5.42768061e-01, -5.93493879e-01,  2.11698562e-01,  4.12660569e-01,
       -1.83748845e-02, -3.60820889e-01, -5.69656268e-02,  3.75402540e-01,
        5.94702065e-01, -3.18870991e-01,  1.19594216e-01, -3.36227119e-02,
        1.72981143e-01, -3.82890046e-01, -4.26839292e-01, -2.82811880e-01,
       -2.98971325e-01, -

In [ ]:
test_df["embedding"][100][100]

np.float32(-0.41012892)

In [ ]:
train_totall = train_totall.reset_index(drop=True)

In [ ]:
train_totall["merged_label"].value_counts()

,count
merged_label,
1,2838724
0,421526
3,317385
2,79313
4,63


In [ ]:
train_totall["embedding"][101]

array([-3.10099095e-01, -3.41315836e-01,  3.79133284e-01, -7.47476339e-01,
       -2.95149982e-01, -1.92298755e-01,  4.51870114e-01,  3.16165864e-01,
        1.00715065e+00,  3.79970223e-01,  2.56055653e-01, -8.46899897e-02,
       -1.06275275e-01,  2.24342346e-01, -1.15649179e-01, -9.74276382e-03,
       -1.09621979e-01, -1.95254266e-01, -2.13412002e-01, -8.07977676e-01,
        6.82970703e-01,  2.36354433e-02,  8.03704321e-01,  2.12491989e-01,
       -1.23257868e-01,  9.57263783e-02,  4.25049029e-02,  4.58052307e-02,
        3.52659106e-01,  4.28587086e-02, -1.20923758e-01, -4.39654291e-01,
       -2.28813410e-01,  2.86091000e-01,  4.93888676e-01, -8.62524379e-03,
       -3.61586183e-01, -3.19923908e-01,  5.34022488e-02, -2.62099266e-01,
       -3.61960202e-01,  8.96944180e-02,  1.41829342e-01,  2.15175539e-01,
        5.20456374e-01, -2.30521113e-01,  5.01014054e-01, -1.34681672e-01,
       -5.28817713e-01, -1.07301705e-01,  1.53642863e-01, -2.81239778e-01,
        1.31683901e-01,  

In [ ]:
X_batch = model.encode(train_totall["sentence"][101], show_progress_bar=False)

In [ ]:
np.array_equal(train_totall["embedding"][101] , X_batch)

False

In [ ]:
X_batch[100]

np.float32(-0.31009915)

In [ ]:
train_totall["embedding"][101][100]

np.float32(-0.080932595)

## 오버샘플링/임베딩
-> 다운샘플링/임베딩
GPU 크기가 너무 많이 듣다.

###다운샘플링
- 1에 맞춰서 다운샘플링하기

In [ ]:
import pandas as pd
from sklearn.utils import resample

# 예: df는 'text'와 'label' 컬럼이 있는 DataFrame
# df = pd.read_csv("your_data.csv")

# 각 라벨의 샘플 수 확인
label_counts = Totall_Data['label'].value_counts()
min_count = 838724  # 가장 적은 수를 기준으로 다운샘플링(1기준)

# 라벨별 다운샘플링
balanced_data = []

for label in label_counts.index:
    subset = Totall_Data[Totall_Data['mer'] == label]
    downsampled = resample(subset,
                           replace=False,
                           n_samples=min_count,
                           random_state=42)
    balanced_data.append(downsampled)

# 결합 및 셔플
balanced_df = pd.concat(balanced_data)
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)


ValueError: Cannot sample 838724 out of arrays with dim 421352 when replace is False

###텍스트 다운로드

In [ ]:
import pandas as pd

# 예시: label 컬럼이 있는 DataFrame
# df = pd.read_csv(...) 또는 pd.read_parquet(...)

# 라벨 1만 샘플링
label_1_sample = Totall_Data[Totall_Data['merged_label'] == 1].sample(n=538724, random_state=42)

# 라벨 0, 2, 3, 4는 전부 다 사용
other_labels = Totall_Data[Totall_Data['merged_label'] != 1]

# 결합
downsampled_df = pd.concat([label_1_sample, other_labels]).reset_index(drop=True)

In [ ]:
downsampled_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1357011 entries, 0 to 1357010
Data columns (total 7 columns):
 #   Column        Non-Null Count    Dtype 
---  ------        --------------    ----- 
 0   Unnamed: 0    1357011 non-null  int64 
 1   sentence      1357011 non-null  object
 2   doc_id        1357011 non-null  int64 
 3   sentence_idx  1357011 non-null  int64 
 4   label         1357011 non-null  int64 
 5   merged_label  1357011 non-null  int64 
 6   embedding     1357011 non-null  object
dtypes: int64(5), object(2)
memory usage: 72.5+ MB


In [ ]:
downsampled_df["merged_label"].value_counts()

,count
merged_label,
1,538724
0,421526
3,317385
2,79313
4,63


In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import random
downsampled_df = pd.read_parquet("/content/drive/MyDrive/DataSet/샘플링_최종_훈련셋/train_totall_set.parquet")

In [2]:
downsampled_df_train = downsampled_df.drop(['sentence', 'label'], axis=1)

###오버샘플링
-> GPU에 과해서 포기

In [ ]:
from imblearn.over_sampling import SMOTE, RandomOverSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sentence_transformers import SentenceTransformer

# 2. 오버샘플링
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X, y)

# 3. 원본 메타정보 추적을 위해 인덱스도 가져옴
resampled_idx = ros.sample_indices_
train_df_resampled = train_df.iloc[resampled_idx].copy()
train_df_resampled['embedding'] = list(X_resampled)
train_df_resampled['label'] = y_resampled

####데이터 로드/저장

테이터 저장

In [ ]:
train_df_resampled.to_parquet("/content/drive/MyDrive/DataSet/suffled_sample/embedding_ko-sroberta.parquet", index=False)

In [ ]:
train_df_resampled = pd.read_parquet("/content/drive/MyDrive/DataSet/suffled_sample/embedding_ko-sroberta.parquet")

In [ ]:
embeddings = train_df_resampled['embedding']
np.save('/content/drive/MyDrive/DataSet/suffled_sample/embedding_X.npy', embeddings)
train_df_resampled.drop(columns='embedding').to_parquet("/content/drive/MyDrive/DataSet/suffled_sample/embedding_ko-sroberta.parquet", index=False)

In [ ]:
train_df_resampled.drop(columns='embedding').to_parquet("/content/drive/MyDrive/DataSet/suffled_sample/embedding_ko-sroberta.parquet", index=False)

데이터 불러오기

In [ ]:
train_df_resampled = pd.read_parquet("/content/drive/MyDrive/DataSet/suffled_sample/embedding_ko-sroberta.parquet")
embeddings = np.load('/content/drive/MyDrive/DataSet/suffled_sample/embedding_X.npy', allow_pickle=True)
train_df_resampled["embedding"] = list(embeddings)

In [ ]:
#train_df_resampled = pd.read_parquet("/content/drive/MyDrive/DataSet/suffled_sample/embedding_ko-sroberta.parquet")
embeddings = np.load('/content/drive/MyDrive/DataSet/suffled_sample/embedding_X.npy', allow_pickle=True)
train_df_resampled["embedding"] = list(embeddings)

## attention모듈 모델 학습 및 평가

In [ ]:
import gc, torch

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

## 모델 적용

In [3]:
downsampled_df_train["merged_label"].value_counts()

,count
merged_label,
1,538724
0,421526
3,317385
2,79313
4,63


In [4]:
downsampled_df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1357011 entries, 0 to 1357010
Data columns (total 5 columns):
 #   Column        Non-Null Count    Dtype 
---  ------        --------------    ----- 
 0   Unnamed: 0    1357011 non-null  int64 
 1   doc_id        1357011 non-null  int64 
 2   sentence_idx  1357011 non-null  int64 
 3   merged_label  1357011 non-null  int64 
 4   embedding     1357011 non-null  object
dtypes: int64(4), object(1)
memory usage: 51.8+ MB


In [5]:
print("라벨 분포 확인:", downsampled_df_train['merged_label'].value_counts())
print("최소/최대 라벨:", downsampled_df_train['merged_label'].min(), downsampled_df_train['merged_label'].max())

라벨 분포 확인: merged_label
1    538724
0    421526
3    317385
2     79313
4        63
Name: count, dtype: int64
최소/최대 라벨: 0 4


In [12]:
# ------------------------------
# 필수 라이브러리 불러오기
# ------------------------------
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from torch.cuda.amp import GradScaler # ✅ 수정됨: autocast는 torch.amp에서 직접 사용
import gc
import os # os.cpu_count() 사용을 위해 추가
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# ------------------------------
# Dataset 클래스
# ------------------------------
class DocumentAwareDataset(Dataset):
    def __init__(self, df):
        self.all_inputs = []
        self.all_labels = []
        self.doc_indexes = []

        # doc_id를 0부터 시작하는 정수 인덱스로 매핑
        unique_doc_ids = df['doc_id'].unique()
        doc_id_map = {doc_id: idx for idx, doc_id in enumerate(unique_doc_ids)}

        for doc_id, group in df.groupby("doc_id"):
            embeddings = np.stack(group['embedding'].values)  # [N, 768] - 사전 계산된 임베딩 사용
            sentence_ids = group['sentence_idx'].values.astype(np.float32)
            labels = group['merged_label'].values.astype(np.int64)

            # 문장 인덱스 정규화 (문서 내 상대적 위치)
            sentence_ids -= sentence_ids.min()
            max_val = sentence_ids.max()
            if len(sentence_ids) > 1 and max_val > 0:
                sentence_ids = sentence_ids / max_val
            else:
                sentence_ids = np.zeros_like(sentence_ids)

            positions = sentence_ids[:, np.newaxis]
            inputs = np.concatenate([embeddings, positions], axis=1) # 임베딩(768) + 위치정보(1) = 769

            doc_index = doc_id_map[doc_id] # 매핑된 정수 인덱스 사용
            for i in range(len(inputs)):
                self.all_inputs.append(inputs[i])
                self.all_labels.append(labels[i])
                self.doc_indexes.append(doc_index)

        self.all_inputs = torch.tensor(np.array(self.all_inputs), dtype=torch.float32)
        self.all_labels = torch.tensor(np.array(self.all_labels), dtype=torch.long)
        self.doc_indexes = torch.tensor(np.array(self.doc_indexes), dtype=torch.long)

    def __len__(self):
        return len(self.all_labels)

    def __getitem__(self, idx):
        return self.all_inputs[idx], self.all_labels[idx], self.doc_indexes[idx]

# ------------------------------
# 모델 정의
# ------------------------------
class AttentiveHierarchicalClassifier(nn.Module):
    def __init__(self, input_dim=769, hidden_dim=256, num_classes=5): # input_dim은 임베딩(768) + 위치(1)
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
        )

        self.attn_doc = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.attn_global = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.classifier = nn.Linear(hidden_dim * 3, num_classes)

    # ✅ 수정됨: AttributeError 해결을 위해 forward 함수 전체 로직 수정
    def forward(self, x, doc_mask):
        h = self.encoder(x)

        doc_context = torch.zeros_like(h)
        # 배치 내 각 문서에 대한 컨텍스트 계산
        for doc_id_val in torch.unique(doc_mask):
            # nonzero()의 결과는 튜플이므로, 첫 번째 요소를 사용해야 합니다.
            indices_tuple = (doc_mask == doc_id_val).nonzero(as_tuple=True)

            # 튜플의 첫 번째 요소(텐서)에 대해 numel()을 호출하여 요소가 있는지 확인합니다.
            if indices_tuple[0].numel() == 0:
                continue

            # 실제 인덱스 텐서를 가져옵니다.
            doc_indices = indices_tuple[0]

            doc_h = h[doc_indices]
            weights = torch.softmax(self.attn_doc(doc_h).squeeze(-1), dim=0)
            weighted = torch.sum(doc_h * weights.unsqueeze(-1), dim=0, keepdim=True)

            # 실제 인덱스 텐서로 컨텍스트를 할당하고, expand 크기도 실제 인덱스 개수를 사용합니다.
            doc_context[doc_indices] = weighted.expand(doc_indices.size(0), -1)

        global_weights = torch.softmax(self.attn_global(h).squeeze(-1), dim=0)
        global_context = torch.sum(h * global_weights.unsqueeze(-1), dim=0, keepdim=True)
        expanded_global = global_context.expand(h.size(0), -1)

        concat = torch.cat([h, doc_context, expanded_global], dim=1)
        logits = self.classifier(concat)
        return logits

# ------------------------------
# 학습 및 평가 함수 + 최고 성능 모델 저장
# ------------------------------
def cross_validate_model(df, n_splits=3, epochs=3, batch_size=128, lr=5e-4):
    X = df.index.values # StratifiedKFold는 인덱스를 사용하도록 변경
    y = df['merged_label'].values
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    all_preds, all_labels_gold = [], []
    best_f1 = -1

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"✅ 현재 사용 중인 디바이스: {device}")
    if torch.cuda.is_available():
        print(f"🛠️ GPU 이름: {torch.cuda.get_device_name(0)}")
        print(f"🔥 초기 GPU 메모리 사용량: {torch.cuda.memory_allocated(device) / 1024**2:.2f} MB")

    available_cpus = os.cpu_count()
    num_workers = min(4, available_cpus if available_cpus is not None else 1)
    print(f"🔩 DataLoader num_workers: {num_workers}")


    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🚀 Fold {fold+1}/{n_splits}")
        train_df = df.iloc[train_idx].reset_index(drop=True)
        val_df = df.iloc[val_idx].reset_index(drop=True)

        train_dataset = DocumentAwareDataset(train_df)
        val_dataset = DocumentAwareDataset(val_df)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                                  num_workers=num_workers, pin_memory=True if device.type == 'cuda' else False,
                                  persistent_workers=True if num_workers > 0 and hasattr(DataLoader, 'persistent_workers') else False,
                                  drop_last=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                                num_workers=num_workers, pin_memory=True if device.type == 'cuda' else False,
                                persistent_workers=True if num_workers > 0 and hasattr(DataLoader, 'persistent_workers') else False)

        model = AttentiveHierarchicalClassifier(num_classes=df['merged_label'].nunique()).to(device)

        if hasattr(torch, 'compile') and device.type == 'cuda':
            print("✅ torch.compile을 사용하여 모델 컴파일 중...")
            try:
                model = torch.compile(model, mode="reduce-overhead")
                print("✅ 모델 컴파일 완료.")
            except Exception as e:
                print(f"⚠️ torch.compile 실패: {e}. 기본 모델로 진행합니다.")

        class_counts = torch.bincount(train_dataset.all_labels)
        epsilon = 1e-6
        class_weights = 1. / (class_counts.float() + epsilon)
        print(f"⚖️ 클래스 가중치: {class_weights}")

        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
        scaler = GradScaler(enabled=device.type == 'cuda')

        model.train()
        for epoch in range(epochs):
            total_loss = 0
            for step, (xb, yb, docb) in enumerate(train_loader):
                xb, yb, docb = xb.to(device), yb.to(device), docb.to(device)

                optimizer.zero_grad(set_to_none=True)

                # ✅ 수정됨: 최신 PyTorch 버전에 맞는 autocast 사용법으로 변경
                with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == 'cuda')):
                    logits = model(xb, docb)
                    loss = criterion(logits, yb)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                total_loss += loss.item()
            print(f"  Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

        model.eval()
        fold_preds, fold_labels_gold = [], []
        with torch.no_grad():
            for xb, yb, docb in val_loader:
                xb, yb, docb = xb.to(device), yb.to(device), docb.to(device)

                # ✅ 수정됨: 최신 PyTorch 버전에 맞는 autocast 사용법으로 변경
                with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == 'cuda')):
                    logits = model(xb, docb)
                preds = torch.argmax(logits, dim=1)
                fold_preds.extend(preds.cpu().numpy())
                fold_labels_gold.extend(yb.cpu().numpy())

        fold_f1 = f1_score(fold_labels_gold, fold_preds, average='macro', zero_division=0)
        print(f"  🎯 Fold {fold+1} Macro-F1: {fold_f1:.4f}")

        if fold_f1 > best_f1:
            best_f1 = fold_f1
            # 컴파일된 모델의 state_dict를 저장하려면 아래와 같이 처리할 수 있습니다.
            # model_state = model._orig_mod.state_dict() if hasattr(model, '_orig_mod') else model.state_dict()
            torch.save(model.state_dict(), 'best_model.pt')
            print("  💾 Best model saved as 'best_model.pt'")

        all_preds.extend(fold_preds)
        all_labels_gold.extend(fold_labels_gold)

        del model, train_loader, val_loader, train_dataset, val_dataset, optimizer, criterion, class_weights
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
            print(f"🔥 Fold 종료 후 GPU 메모리 사용량: {torch.cuda.memory_allocated(device) / 1024**2:.2f} MB")

    acc = accuracy_score(all_labels_gold, all_preds)
    final_f1 = f1_score(all_labels_gold, all_preds, average='macro', zero_division=0)
    print(f"\n✅ Cross-Validation Accuracy: {acc:.4f}, Macro-F1: {final_f1:.4f}")
    print("🎉 Best Fold Macro-F1: {:.4f}".format(best_f1))
    print("📦 Saved best model to: best_model.pt")

    print("\n📋 최종 분류 리포트:")
    print(classification_report(all_labels_gold, all_preds, zero_division=0))

In [13]:
cross_validate_model(downsampled_df_train, n_splits=2, epochs=5, batch_size=4098, lr=1e-4)

✅ 현재 사용 중인 디바이스: cuda
🛠️ GPU 이름: NVIDIA A100-SXM4-40GB
🔥 초기 GPU 메모리 사용량: 29.19 MB
🔩 DataLoader num_workers: 4

🚀 Fold 1/2
✅ torch.compile을 사용하여 모델 컴파일 중...
✅ 모델 컴파일 완료.
⚖️ 클래스 가중치: tensor([4.7447e-06, 3.7125e-06, 2.5216e-05, 6.3015e-06, 3.2258e-02])


<ipython-input-12-1541135362>:176: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=device.type == 'cuda')


  Epoch 1/2 | Loss: 0.6790
  Epoch 2/2 | Loss: 0.3714
  🎯 Fold 1 Macro-F1: 0.6493
  💾 Best model saved as 'best_model.pt'
🔥 Fold 종료 후 GPU 메모리 사용량: 29.49 MB

🚀 Fold 2/2
✅ torch.compile을 사용하여 모델 컴파일 중...
✅ 모델 컴파일 완료.
⚖️ 클래스 가중치: tensor([4.7447e-06, 3.7125e-06, 2.5217e-05, 6.3015e-06, 3.1250e-02])


<ipython-input-12-1541135362>:176: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=device.type == 'cuda')


  Epoch 1/2 | Loss: 0.6481
  Epoch 2/2 | Loss: 0.3639
  🎯 Fold 2 Macro-F1: 0.6453
🔥 Fold 종료 후 GPU 메모리 사용량: 29.49 MB

✅ Cross-Validation Accuracy: 0.8485, Macro-F1: 0.6473
🎉 Best Fold Macro-F1: 0.6493
📦 Saved best model to: best_model.pt

📋 최종 분류 리포트:
              precision    recall  f1-score   support

           0       0.98      0.95      0.97    421526
           1       0.96      0.68      0.80    538724
           2       0.41      0.96      0.57     79313
           3       0.83      0.96      0.89    317385
           4       0.00      0.65      0.01        63

    accuracy                           0.85   1357011
   macro avg       0.64      0.84      0.65   1357011
weighted avg       0.90      0.85      0.86   1357011

